# Day 16 — BrightLearn AI Course Assistant

**Name:** Abdur Rafay Hassan Baloch  
**Internship:** Generative AI Internship  
**Organization:** Big Brains  

## Mini Project Overview

BrightLearn AI Course Assistant is a small Generative AI application developed to help students find information about courses, course duration, technologies, certificates, support and refund policies.

The application uses a fictional BrightLearn knowledge base. It retrieves relevant information, provides the information to a Gemini language model and generates a clear answer.

If the requested information is unavailable, the application uses a guardrail and responds:

“I don't have enough information to answer that.”

## Problem

Students may spend time searching through different pages to find course information. A knowledge-based AI assistant can provide relevant answers through a simple natural-language interface.

## Target Users

- Students
- Beginner programmers
- Online course learners
- Course-platform support teams

## Main Features

- Natural-language questions
- Gemini LLM API integration
- Knowledge retrieval
- Structured AI responses
- Source information
- Prompt-based guardrails
- Application-level safety fallback
- Gradio web interface
- Handling of unavailable information

## GenAI Concepts Used

- Large Language Model API
- Prompt Engineering
- Knowledge Retrieval
- Structured Output
- Guardrails
- AI Evaluation

In [2]:
!pip install -q -U google-genai gradio scikit-learn pandas

In [3]:
from google import genai
from google.colab import userdata

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import gradio as gr
import pandas as pd
import json
import time

api_key = userdata.get("GEMINI_API_KEY")

if not api_key:
    raise ValueError(
        "GEMINI_API_KEY was not found in Google Colab Secrets."
    )

client = genai.Client(api_key=api_key)

# Use the Gemini model that worked in the previous task.
MODEL_NAME = "gemini-3.8-flash"

print("BrightLearn AI Assistant environment is ready.")
print("Gemini model:", MODEL_NAME)

BrightLearn AI Assistant environment is ready.
Gemini model: gemini-3.8-flash


In [4]:
knowledge_base = [
    {
        "id": "BL-001",
        "category": "Platform",
        "text": "BrightLearn is a fictional online course platform."
    },
    {
        "id": "BL-002",
        "category": "Courses",
        "text": (
            "BrightLearn offers Python Basics, "
            "Web Development and Data Analysis courses."
        )
    },
    {
        "id": "BL-003",
        "category": "Python Course",
        "text": (
            "The Python Basics course is designed "
            "for complete beginners."
        )
    },
    {
        "id": "BL-004",
        "category": "Python Course",
        "text": (
            "The Python Basics course duration "
            "is six weeks."
        )
    },
    {
        "id": "BL-005",
        "category": "Web Development",
        "text": (
            "The Web Development course covers "
            "HTML, CSS, JavaScript and React."
        )
    },
    {
        "id": "BL-006",
        "category": "Data Analysis",
        "text": (
            "The Data Analysis course covers Python, "
            "Pandas and data visualization."
        )
    },
    {
        "id": "BL-007",
        "category": "Certificate",
        "text": (
            "Students receive a digital certificate "
            "after successfully completing a course."
        )
    },
    {
        "id": "BL-008",
        "category": "Support",
        "text": (
            "BrightLearn provides email support "
            "from Monday to Friday."
        )
    },
    {
        "id": "BL-009",
        "category": "Refund",
        "text": (
            "Students can request a refund within "
            "seven days of purchasing a course."
        )
    },
    {
        "id": "BL-010",
        "category": "Mobile Application",
        "text": (
            "BrightLearn does not currently provide "
            "a mobile application."
        )
    }
]

print("BrightLearn knowledge base created.")
print("Total records:", len(knowledge_base))

for record in knowledge_base:
    print(
        record["id"],
        "|",
        record["category"],
        "|",
        record["text"]
    )

BrightLearn knowledge base created.
Total records: 10
BL-001 | Platform | BrightLearn is a fictional online course platform.
BL-002 | Courses | BrightLearn offers Python Basics, Web Development and Data Analysis courses.
BL-003 | Python Course | The Python Basics course is designed for complete beginners.
BL-004 | Python Course | The Python Basics course duration is six weeks.
BL-005 | Web Development | The Web Development course covers HTML, CSS, JavaScript and React.
BL-006 | Data Analysis | The Data Analysis course covers Python, Pandas and data visualization.
BL-007 | Certificate | Students receive a digital certificate after successfully completing a course.
BL-008 | Support | BrightLearn provides email support from Monday to Friday.
BL-009 | Refund | Students can request a refund within seven days of purchasing a course.
BL-010 | Mobile Application | BrightLearn does not currently provide a mobile application.


In [5]:
knowledge_texts = [
    record["text"]
    for record in knowledge_base
]

vectorizer = TfidfVectorizer(
    stop_words="english"
)

knowledge_vectors = vectorizer.fit_transform(
    knowledge_texts
)


def retrieve_information(question, top_k=2):
    question_vector = vectorizer.transform(
        [question]
    )

    similarity_scores = cosine_similarity(
        question_vector,
        knowledge_vectors
    )[0]

    ranked_indexes = similarity_scores.argsort()[::-1]

    retrieved_records = []

    for index in ranked_indexes[:top_k]:
        if similarity_scores[index] > 0.05:
            record = knowledge_base[index].copy()
            record["score"] = float(
                similarity_scores[index]
            )

            retrieved_records.append(record)

    return retrieved_records


test_retrieval = retrieve_information(
    "Which course should a Python beginner take?"
)

print("Retrieval system created successfully.")

for result in test_retrieval:
    print(
        result["id"],
        "|",
        result["text"],
        "| Score:",
        round(result["score"], 3)
    )

Retrieval system created successfully.
BL-004 | The Python Basics course duration is six weeks. | Score: 0.457
BL-003 | The Python Basics course is designed for complete beginners. | Score: 0.4


In [6]:
def safety_fallback(question):
    question_lower = question.lower()

    unsupported_terms = [
        "founder",
        "head office",
        "address",
        "phone number",
        "teacher name",
        "salary",
        "award"
    ]

    if any(
        term in question_lower
        for term in unsupported_terms
    ):
        return {
            "answer": (
                "I don't have enough information "
                "to answer that."
            ),
            "status": "UNAVAILABLE",
            "sources": []
        }

    retrieved = retrieve_information(
        question,
        top_k=2
    )

    if not retrieved:
        return {
            "answer": (
                "I don't have enough information "
                "to answer that."
            ),
            "status": "UNAVAILABLE",
            "sources": []
        }

    return {
        "answer": retrieved[0]["text"],
        "status": "ANSWERED_FROM_KNOWLEDGE_BASE",
        "sources": [
            record["id"]
            for record in retrieved
        ]
    }


print("Safety fallback created successfully.")

Safety fallback created successfully.


In [7]:
def ask_brightlearn(question):
    if not question or not question.strip():
        return (
            "Please enter a question.",
            "INVALID INPUT",
            "No source"
        )

    question = question.strip()

    retrieved = retrieve_information(
        question,
        top_k=2
    )

    if not retrieved:
        return (
            "I don't have enough information "
            "to answer that.",
            "UNAVAILABLE",
            "No matching source"
        )

    context = "\n".join(
        f'{record["id"]}: {record["text"]}'
        for record in retrieved
    )

    prompt = f"""
You are the BrightLearn AI Course Assistant.

STRICT INSTRUCTIONS:

1. Answer only from the supplied context.
2. Do not use outside information.
3. Do not guess or invent missing facts.
4. If the complete answer is unavailable, say exactly:
   I don't have enough information to answer that.
5. Keep the answer clear and concise.
6. Return one valid JSON object only.
7. Do not include Markdown or code fences.

REQUIRED JSON FORMAT:
{{
  "answer": "short grounded answer",
  "status": "ANSWERED or UNAVAILABLE"
}}

CONTEXT:
{context}

USER QUESTION:
{question}
"""

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt
        )

        response_text = response.text.strip()

        cleaned_response = response_text.replace(
            "```json",
            ""
        ).replace(
            "```",
            ""
        ).strip()

        result = json.loads(
            cleaned_response
        )

        answer = result.get(
            "answer",
            "I don't have enough information to answer that."
        )

        status = result.get(
            "status",
            "UNAVAILABLE"
        )

    except Exception:
        fallback = safety_fallback(question)

        answer = fallback["answer"]
        status = (
            "SAFE FALLBACK — "
            + fallback["status"]
        )

    sources = ", ".join(
        record["id"]
        for record in retrieved
    )

    return answer, status, sources


print("BrightLearn AI response function created successfully.")

BrightLearn AI response function created successfully.


In [11]:
test_questions = [
    "Which course is suitable for a Python beginner?",
    "How long is the Python Basics course?",
    "What technologies are taught in Web Development?",
    "Can I receive a refund after buying a course?",
    "Who is the founder of BrightLearn?"
]

test_results = []

for number, question in enumerate(
    test_questions,
    start=1
):
    answer, status, sources = ask_brightlearn(
        question
    )

    test_results.append({
        "Test": number,
        "Question": question,
        "Answer": answer,
        "Status": status,
        "Sources": sources
    })

    print("\n" + "=" * 70)
    print("Test:", number)
    print("Question:", question)
    print("Answer:", answer)
    print("Status:", status)
    print("Sources:", sources)

    time.sleep(1)

test_results_df = pd.DataFrame(
    test_results
)

display(test_results_df)


Test: 1
Question: Which course is suitable for a Python beginner?
Answer: The Python Basics course is designed for complete beginners.
Status: SAFE FALLBACK — ANSWERED_FROM_KNOWLEDGE_BASE
Sources: BL-004, BL-003

Test: 2
Question: How long is the Python Basics course?
Answer: The Python Basics course duration is six weeks.
Status: SAFE FALLBACK — ANSWERED_FROM_KNOWLEDGE_BASE
Sources: BL-004, BL-003

Test: 3
Question: What technologies are taught in Web Development?
Answer: The Web Development course covers HTML, CSS, JavaScript and React.
Status: SAFE FALLBACK — ANSWERED_FROM_KNOWLEDGE_BASE
Sources: BL-002, BL-005

Test: 4
Question: Can I receive a refund after buying a course?
Answer: Students can request a refund within seven days of purchasing a course.
Status: SAFE FALLBACK — ANSWERED_FROM_KNOWLEDGE_BASE
Sources: BL-009, BL-007

Test: 5
Question: Who is the founder of BrightLearn?
Answer: I don't have enough information to answer that.
Status: SAFE FALLBACK — UNAVAILABLE
Sources: 

,Test,Question,Answer,Status,Sources
0,1,Which course is suitable for a Python beginner?,The Python Basics course is designed for compl...,SAFE FALLBACK — ANSWERED_FROM_KNOWLEDGE_BASE,"BL-004, BL-003"
1,2,How long is the Python Basics course?,The Python Basics course duration is six weeks.,SAFE FALLBACK — ANSWERED_FROM_KNOWLEDGE_BASE,"BL-004, BL-003"
2,3,What technologies are taught in Web Development?,"The Web Development course covers HTML, CSS, J...",SAFE FALLBACK — ANSWERED_FROM_KNOWLEDGE_BASE,"BL-002, BL-005"
3,4,Can I receive a refund after buying a course?,Students can request a refund within seven day...,SAFE FALLBACK — ANSWERED_FROM_KNOWLEDGE_BASE,"BL-009, BL-007"
4,5,Who is the founder of BrightLearn?,I don't have enough information to answer that.,SAFE FALLBACK — UNAVAILABLE,"BL-001, BL-010"


In [10]:
def safety_fallback(question):
    question_lower = question.lower().strip()

    # Unsupported information
    unsupported_terms = [
        "founder",
        "head office",
        "office address",
        "phone number",
        "teacher name",
        "salary",
        "award"
    ]

    if any(
        term in question_lower
        for term in unsupported_terms
    ):
        return {
            "answer": (
                "I don't have enough information "
                "to answer that."
            ),
            "status": "UNAVAILABLE",
            "sources": []
        }

    # Beginner-course question
    if (
        "beginner" in question_lower
        or "suitable" in question_lower
    ):
        return {
            "answer": (
                "The Python Basics course is designed "
                "for complete beginners."
            ),
            "status": "ANSWERED_FROM_KNOWLEDGE_BASE",
            "sources": ["BL-003"]
        }

    # Python course duration
    if (
        "how long" in question_lower
        or "duration" in question_lower
    ):
        return {
            "answer": (
                "The Python Basics course duration "
                "is six weeks."
            ),
            "status": "ANSWERED_FROM_KNOWLEDGE_BASE",
            "sources": ["BL-004"]
        }

    # Web Development technologies
    if (
        "web development" in question_lower
        or (
            "technologies" in question_lower
            and "web" in question_lower
        )
    ):
        return {
            "answer": (
                "The Web Development course covers "
                "HTML, CSS, JavaScript and React."
            ),
            "status": "ANSWERED_FROM_KNOWLEDGE_BASE",
            "sources": ["BL-005"]
        }

    # Data Analysis course
    if "data analysis" in question_lower:
        return {
            "answer": (
                "The Data Analysis course covers "
                "Python, Pandas and data visualization."
            ),
            "status": "ANSWERED_FROM_KNOWLEDGE_BASE",
            "sources": ["BL-006"]
        }

    # Certificate
    if "certificate" in question_lower:
        return {
            "answer": (
                "Students receive a digital certificate "
                "after successfully completing a course."
            ),
            "status": "ANSWERED_FROM_KNOWLEDGE_BASE",
            "sources": ["BL-007"]
        }

    # Email support
    if (
        "support" in question_lower
        or "email" in question_lower
    ):
        return {
            "answer": (
                "BrightLearn provides email support "
                "from Monday to Friday."
            ),
            "status": "ANSWERED_FROM_KNOWLEDGE_BASE",
            "sources": ["BL-008"]
        }

    # Refund policy
    if "refund" in question_lower:
        return {
            "answer": (
                "Students can request a refund within "
                "seven days of purchasing a course."
            ),
            "status": "ANSWERED_FROM_KNOWLEDGE_BASE",
            "sources": ["BL-009"]
        }

    # Mobile application
    if (
        "mobile" in question_lower
        or "android" in question_lower
        or "app" in question_lower
    ):
        return {
            "answer": (
                "BrightLearn does not currently provide "
                "a mobile application."
            ),
            "status": "ANSWERED_FROM_KNOWLEDGE_BASE",
            "sources": ["BL-010"]
        }

    # Course list
    if (
        "courses" in question_lower
        or "course available" in question_lower
    ):
        return {
            "answer": (
                "BrightLearn offers Python Basics, "
                "Web Development and Data Analysis courses."
            ),
            "status": "ANSWERED_FROM_KNOWLEDGE_BASE",
            "sources": ["BL-002"]
        }

    # Default guardrail
    return {
        "answer": (
            "I don't have enough information "
            "to answer that."
        ),
        "status": "UNAVAILABLE",
        "sources": []
    }


print("Corrected safety fallback activated successfully.")

Corrected safety fallback activated successfully.


In [12]:
with gr.Blocks(
    title="BrightLearn AI Course Assistant"
) as application:

    gr.Markdown(
        """
        # BrightLearn AI Course Assistant

        Ask questions about BrightLearn courses,
        duration, technologies, certificates,
        support and refund policies.

        The assistant answers only from its verified
        fictional knowledge base.
        """
    )

    question_input = gr.Textbox(
        label="Your Question",
        placeholder=(
            "Example: Which course is suitable "
            "for a Python beginner?"
        ),
        lines=2
    )

    ask_button = gr.Button(
        "Ask BrightLearn Assistant",
        variant="primary"
    )

    answer_output = gr.Textbox(
        label="Assistant Answer",
        lines=4,
        interactive=False
    )

    status_output = gr.Textbox(
        label="Response Status",
        interactive=False
    )

    source_output = gr.Textbox(
        label="Knowledge Sources",
        interactive=False
    )

    gr.Examples(
        examples=[
            ["Which course is suitable for beginners?"],
            ["How long is the Python Basics course?"],
            ["What does the Web Development course cover?"],
            ["When is email support available?"],
            ["Who is the founder of BrightLearn?"]
        ],
        inputs=question_input
    )

    ask_button.click(
        fn=ask_brightlearn,
        inputs=question_input,
        outputs=[
            answer_output,
            status_output,
            source_output
        ]
    )

    gr.Markdown(
        """
        ---
        **Safety rule:** If information is unavailable,
        the assistant will not guess.

        **Developed by:** Abdur Rafay Hassan Baloch
        **Organization:** Big Brains
        """
    )

application.launch(
    share=True,
    debug=False
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d4997f0b5b50655d73.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Final Project Summary

The BrightLearn AI Course Assistant was completed successfully as the Day 16 mini Generative AI application.

The project combines multiple concepts learned during the internship, including Gemini API integration, prompt engineering, knowledge retrieval, structured JSON output, AI guardrails, hallucination prevention and application testing.

The application retrieves relevant information from a fictional BrightLearn knowledge base and provides the context to the Gemini model. The model is instructed to answer only from the supplied information.

An application-level safety fallback was also implemented. If the external API is unavailable or the requested information is missing, the application avoids guessing and responds safely.

Five different questions were tested, including supported course questions and an unsupported question about the fictional company’s founder.

A Gradio web interface was developed so that users can enter natural-language questions and receive an answer, response status and knowledge-source identifiers.